In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import sys
sys.path.append("../")

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-dark')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import rateslib as rl

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

In [2]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from SDRUtils.products.usd.usd_swaptions import USD_Swaptions
from SDRUtils.products._swaptions.pricer import (
    usd_swaption_straddle_pricer_from_row,
    usd_swaption_leg_pricer_from_row,
    usd_swaption_dealer_risk_reversal_skew_from_row,
    USDSwaptionStraddlePricerResult,
    USDSwaptionLegPricerResult,
	USDSwaptionDealerRiskReversalSkewResult,
    _compute_swaption_leg_greeks,
    USDSwaptionVerticalSpreadPricerResult,
    usd_swaption_vertical_spread_pricer_from_row
)

In [3]:
cache_path = r"C:\Users\chris\clee\project-oasis\private\sdranalytics\.cache"

as_of = datetime.date(2026, 1, 15)
start = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 0, 0))
end = NY_tz.localize(datetime.datetime(as_of.year, as_of.month, as_of.day, 23, 59))

mdp = IRSwapsMDP(source="ERIS_EOD_LIVE-QL_BASIC")
pricer = mdp.get_pricer(request=dict(curve_name="USD-SOFR-1D", timestamp=start.date()))

from SDRUtils.data.builder import SDRDataBuilder
sdr = SDRDataBuilder(cache_path=cache_path, show_tqdm=True)
df = sdr.grab_sdr_trades(
	start_timestamp=start,
	end_timestamp=end,
	agency="CFTC",
	asset_class="RATES",
)
# df

MERGING SLICES...: 100%|██████████| 2/2 [00:00<00:00, 88.82it/s]


In [11]:
# sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path)
sdf = USD_Swaptions().build_classification_dataframe(start=start, end=end, cache_path=cache_path, ignore_cache=True, merge_package_legs=False)
sdf

PRICING VERTICAL SPREADS...:  24%|██▍       | 6/25 [00:00<00:00, 50.13it/s]WARNING	Task(Task-3) SDRUtils.packages.swaption_packages:swaption_packages.py:detect_and_link_swaption_packages_df()- Failed to price vertical spread package VERTICAL_SPREAD_1x1_0ba57dbee458: instrument expired
WARNING	Task(Task-3) SDRUtils.packages.swaption_packages:swaption_packages.py:detect_and_link_swaption_packages_df()- Failed to price vertical spread package VERTICAL_SPREAD_1x1_f2344d16e9d4: root not bracketed: f[0,1] -> [1.983387e+06,3.313343e+08]
WARNING	Task(Task-3) SDRUtils.packages.swaption_packages:swaption_packages.py:detect_and_link_swaption_packages_df()- Failed to price vertical spread package VERTICAL_SPREAD_1x1_79cd9e409cea: root not bracketed: f[0,1] -> [1.776582e+06,3.882652e+08]
WARNING	Task(Task-3) SDRUtils.packages.swaption_packages:swaption_packages.py:detect_and_link_swaption_packages_df()- Failed to price vertical spread package VERTICAL_SPREAD_1x1_97532dee78a1: root not bracketed: f[

,event_action,trade_id,execution_timestamp,effective_date,expiration_date,product_type,trade_label,notional,notional_currency,is_notional_capped,...,vs_theta1d,vs_atm_strike_offset,vs_otm_strike_offset,vega_curve_type,vega_curve_id,vega_curve_legs,vega_curve_vega01,vega_curve_weight,vega_curve_vega_ratio,vega_curve_pricing_method
0,MODI-TRAD,1732643977000000201,2026-01-15 05:13:03+00:00,2024-01-17,2026-01-20,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 2Yx1Y PAYER EURO...,120000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
1,MODI-TRAD,1732644545000000101,2026-01-15 05:13:34+00:00,2024-01-17,2026-01-20,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 2Yx1Y PAYER ...,120000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
2,MODI-TRAD,1733609362000000101,2026-01-15 07:21:02+00:00,2026-01-14,2026-05-19,SWAPTION_PAYER,USD-SOFR 1D CUSTOM 4Mx10Y PAYER EURO VANILLA CASH,250000000.0,USD,True,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
3,MODI-TRAD,1733609363000000201,2026-01-15 07:21:02+00:00,2026-01-14,2026-05-19,SWAPTION_RECEIVER,USD-SOFR 1D CUSTOM 4Mx10Y RECEIVER EURO VANILL...,250000000.0,USD,True,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
4,NEWT-TRAD,1737518263000001001,2026-01-15 09:51:55+00:00,2026-01-15,2027-01-15,SWAPTION_PAYER,USD-SOFR-COMPOUND 1D CONSTANT 1Yx10Y PAYER EUR...,50000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
471,NEWT-NOVA,1746865872000000401,2026-01-15 22:14:58+00:00,2026-01-15,2027-09-20,SWAPTION_RECEIVER,USD-SOFR-OIS Compound 1D CONSTANT 1Y8MxIMM_U20...,130000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
484,NEWT-TRAD,1746962551000000101,2026-01-15 22:43:33+00:00,2022-10-06,2026-01-16,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 3Y3Mx30Y PAY...,160000000.0,USD,True,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
485,TERM-ETRM,1746962552000000201,2026-01-15 22:43:33+00:00,2022-10-06,2026-01-16,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 3Y3Mx30Y PAY...,160000000.0,USD,True,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None
486,NEWT-NOVA,1746962553000000301,2026-01-15 22:43:33+00:00,2025-07-28,2026-01-16,SWAPTION_PAYER,USD-SOFR-OIS Compound 1D CONSTANT 6Mx10Y PAYER...,60000000.0,USD,False,...,NaN,NaN,NaN,None,None,None,NaN,NaN,NaN,None


In [36]:
sdf.columns

Index(['event_action', 'trade_id', 'execution_timestamp', 'effective_date',
       'expiration_date', 'product_type', 'trade_label', 'notional',
       'notional_currency', 'is_notional_capped', 'package_type', 'package_id',
       'package_legs', 'underlying_expiration_date', 'tenor_years',
       'tenor_label', 'forward_start_years', 'forward_label', 'premium',
       'exercise_style', 'strike', 'upi_underlier_name',
       'unique_product_identifier', 'platform_identifier', 'cleared',
       'package_indicator', 'package_transaction_price',
       'option_premium_amount', 'package_confidence', 'package_reason',
       'package_legs_count', 'rr_atmf', 'rr_out_strike', 'rr_skew_bpvol',
       'rr_atm_bpvol', 'rr_payer_skew', 'rr_receiver_skew', 'rr_dv01',
       'rr_wing_dv01', 'rr_gamma01', 'rr_vega01', 'rr_theta1d',
       'custy_rr_width_bps', 'straddle_bpvol_yr', 'straddle_fwd_premium',
       'straddle_dv01', 'straddle_vega01', 'straddle_gamma01',
       'straddle_theta1d', 'ladd

In [52]:
# sdf["package_type"].value_counts()

sdf[(sdf["trade_label"].str.contains("x")) & (sdf["ladder_strikes"].notna())][
    ["trade_id", "execution_timestamp", "trade_label", "notional", "strike", "premium", "platform_identifier"]
]
# sdf[sdf["trade_label"].str.contains("5Yx10Y")][["trade_id", "execution_timestamp", "trade_label", "notional", "strike", "premium", "platform_identifier"]]

# sdf[sdf["package_type"].str.contains("CUSTY")].head(4)

# temp = sdf[sdf["vs_atm_bpvol_yr"].notna()].head(24)
# temp["execution_timestamp"] = temp["execution_timestamp"].astype(str)
# temp.to_excel("2026-01-15_sdr_vs.xlsx")

# sdf[(sdf["package_type"].str.contains("RISK_R")) & (sdf["trade_label"].str.contains("2Yx10Y"))].to_dict(orient="records")
# temp.to_excel("2026-01-15_sdr_swaption.xlsx")

,trade_id,execution_timestamp,trade_label,notional,strike,premium,platform_identifier
39,1741981511000002201,2026-01-15 13:03:30+00:00,USD-SOFR-OIS Compound 1D CONSTANT 6Mx30Y RECEI...,200000000.0,0.0380,63.25,XXXX
40,1741996824000000101,2026-01-15 13:03:30+00:00,USD-SOFR-OIS Compound 1D CONSTANT 6Mx30Y RECEI...,100000000.0,0.0370,63.25,XXXX
38,1741968482000002001,2026-01-15 13:03:30+00:00,USD-SOFR-OIS Compound 1D CONSTANT 6Mx30Y RECEI...,100000000.0,0.0400,632500.00,XXXX
257,1744567123000001501,2026-01-15 17:46:22+00:00,USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y RECEI...,50000000.0,0.0350,100000.00,BILT
256,1744567122000001401,2026-01-15 17:46:22+00:00,USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y RECEI...,50000000.0,0.0310,60000.00,BILT
255,1744567121000001301,2026-01-15 17:46:22+00:00,USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y RECEI...,50000000.0,0.0390,160000.00,BILT
258,1744567124000001601,2026-01-15 17:46:23+00:00,USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y RECEI...,50000000.0,0.0310,60000.00,BILT
259,1744567125000001701,2026-01-15 17:46:25+00:00,USD-SOFR-OIS Compound 1D CONSTANT 3Yx10Y RECEI...,50000000.0,0.0390,160000.00,BILT
302,1745290227000000301,2026-01-15 18:55:26+00:00,USD-SOFR-OIS Compound 1D CONSTANT 6Mx2Y RECEIV...,270000000.0,0.0235,0.00,XXXX
303,1745294220000000101,2026-01-15 18:55:26+00:00,USD-SOFR-OIS Compound 1D CONSTANT 6Mx2Y RECEIV...,800000000.0,0.0235,0.00,XXXX


In [18]:
import ujson as json


def format_swaption_pricing_results(
    results: USDSwaptionStraddlePricerResult | USDSwaptionLegPricerResult | USDSwaptionDealerRiskReversalSkewResult,
):
    if isinstance(results, USDSwaptionDealerRiskReversalSkewResult):
        output = {
            "trade": results.trade_label,
            "atm_strike": results.atm_strike * 100,
            "otm_payer_strike": results.otm_payer_strike * 100,
            "otm_receiver_strike": results.otm_receiver_strike * 100,
            "wing_strike_width": results.wing_strike_width,
            "atm_bpvol": results.atm_bpvol_yr,
            "otm_payer_bpvol": results.otm_payer_bpvol_yr,
            "otm_receiver_bpvol": results.otm_receiver_bpvol_yr,
            "payer_skew_bpvol_yr": results.payer_skew_bpvol_yr,
            "receiver_skew_bpvol_yr": results.receiver_skew_bpvol_yr,
            "skew_bpvol": results.skew_bpvol_yr,
            "atm_notional": results.atm_notional,
            "wing_notional": results.wing_notional,
            "otm_payer_vega01": results.otm_payer_vega01,
            "otm_receiver_vega01": results.otm_receiver_vega01,
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
            "wing_dv01": results.wing_dv01
        }
    elif isinstance(results, USDSwaptionVerticalSpreadPricerResult):
        output = {
            "trade": results.trade_label,
            "spread_type": results.spread_type,
            # Strikes
            "atm_strike": results.atm_strike * 100,
            "otm_strike": results.otm_strike * 100,
            "strike_width_bps": results.strike_width_bps,
            "atm_strike_offset": results.atm_strike_offset,
            "otm_strike_offset": results.otm_strike_offset,
            # Vols
            "atm_bpvol": results.atm_bpvol_yr,
            "otm_bpvol": results.otm_bpvol_yr,
            "vol_spread_bpvol": results.vol_spread_bpvol_yr,
            # Notionals
            "atm_notional": results.atm_notional,
            "otm_notional": results.otm_notional,
            "notional_ratio": results.notional_ratio,
            # Premiums
            "net_premium": results.net_premium,
            "atm_premium": results.atm_premium,
            "otm_premium": results.otm_premium,
            # ATM leg Greeks
            "atm_dv01": results.atm_dv01,
            "atm_gamma01": results.atm_gamma01,
            "atm_vega01": results.atm_vega01,
            
            "atm_theta1d": results.atm_theta1d,
            # OTM leg Greeks
            "otm_dv01": results.otm_dv01,
            "otm_gamma01": results.otm_gamma01,
            "otm_vega01": results.otm_vega01,
            "otm_theta1d": results.otm_theta1d,
            # Aggregate Greeks
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
        }
    else:
        output = {
            "trade": results.trade_label,
            "prem": (results.fwd_prem / results.notional) * 10_000,
            "bpvol": results.bpvol_yr,
            "bpvol_day": results.bpvol_yr / np.sqrt(252),
            "dv01": results.dv01,
            "gamma01": results.gamma01,
            "vega01": results.vega01,
            "theta1d": results.theta1d,
        }

    print(json.dumps(output, indent=4))

In [50]:
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(sdf.loc[168], pricer))
format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(sdf.loc[240], pricer))

{
    "trade": "USD-SOFR-OIS Compound 1D CONSTANT 5Yx10Y RECEIVER EURO VANILLA PHYS",
    "prem": 1112.0,
    "bpvol": 77.05838128123648,
    "bpvol_day": 4.854221745317774,
    "dv01": -15.299188158953257,
    "gamma01": 73.86230753604254,
    "vega01": 30475.18649504029,
    "theta1d": -643.1243555208202
}


In [18]:
# format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(sdf.loc[199], pricer))
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(sdf.loc[96], pricer))

# format_swaption_pricing_results(usd_swaption_dealer_risk_reversal_skew_from_row(risk_reversal_row=sdf.loc[286], pricer=pricer))

In [53]:

row = sdf.loc[168]

display(row.to_dict())
_compute_swaption_leg_greeks(
	pricer,
	row["expiration_date"],
	row["underlying_expiration_date"],
	row["strike"],
	row["notional"],
	row["premium"],
	"receiver" if "rec" in row["product_type"].lower() else "payer",
)

{'event_action': 'NEWT-TRAD',
 'trade_id': '1745734985000000401',
 'execution_timestamp': Timestamp('2026-01-15 20:56:16+0000', tz='UTC'),
 'effective_date': Timestamp('2026-01-15 00:00:00'),
 'expiration_date': Timestamp('2028-01-10 00:00:00'),
 'product_type': 'SWAPTION_PAYER',
 'trade_label': 'USD-SOFR-OIS Compound 1Y CONSTANT 2Yx30Y PAYER EURO VANILLA PHYS',
 'notional': 100000000.0,
 'notional_currency': 'USD',
 'is_notional_capped': False,
 'package_type': 'SWAPTION',
 'package_id': None,
 'package_legs': None,
 'underlying_expiration_date': Timestamp('2058-01-12 00:00:00'),
 'tenor_years': 30.027397260273972,
 'tenor_label': '30Y',
 'forward_start_years': 1.9863013698630136,
 'forward_label': '2Y',
 'premium': 450000.0,
 'exercise_style': 'EUROPEAN',
 'strike': 0.05234,
 'upi_underlier_name': 'NA/Swap OIS USD',
 'unique_product_identifier': 'QZZLNQ2D4JQT',
 'platform_identifier': 'BILT',
 'cleared': 'N',
 'package_indicator': False,
 'package_transaction_price': '',
 'option_pre

_SwaptionLegGreeks(bpvol_yr=50.80008511489668, dv01=13146.336552531715, gamma01=394.24023436883004, vega01=34351.294088425864, theta1d=1203.1005319558317, strike_offset=100)

In [452]:
# temp = sdf.loc[422].copy()

# temp["premium"] = temp["premium"] / 4
# format_swaption_pricing_results(usd_swaption_leg_pricer_from_row(temp, pricer))
# format_swaption_pricing_results(usd_swaption_straddle_pricer_from_row(temp, pricer))

In [123]:
# ids = [1712299926000000501, 1712299925000000401]

ids = [
1745547082000000201,
1745547081000000101


]

# df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_csv("_temp_raw_raw_trades.csv",index=False)

df[df["Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df[df["Original Dissemination Identifier"].isin([str(id) for id in ids])].to_dict(orient="records")
# df["Action type"].value_counts()
# df["Event type"].value_counts()

[{'Dissemination Identifier': '1745547082000000201',
  'Original Dissemination Identifier': '',
  'Action type': 'NEWT',
  'Event type': 'TRAD',
  'Event timestamp': Timestamp('2026-01-15 19:23:44+0000', tz='UTC'),
  'Amendment indicator': None,
  'Asset Class': 'IR',
  'Product name': None,
  'Cleared': 'N',
  'Mandatory clearing indicator': False,
  'Execution Timestamp': Timestamp('2026-01-15 19:23:44+0000', tz='UTC'),
  'Effective Date': Timestamp('2026-01-15 00:00:00'),
  'Expiration Date': Timestamp('2026-02-17 00:00:00'),
  'Maturity date of the underlier': datetime.date(2056, 2, 19),
  'Non-standardized term indicator': False,
  'Platform identifier': 'BILT',
  'Prime brokerage transaction indicator': False,
  'Block trade election indicator': False,
  'Large notional off-facility swap election indicator': False,
  'Notional amount-Leg 1': '60,000,000',
  'Notional amount-Leg 2': '60,000,000',
  'Notional currency-Leg 1': 'USD',
  'Notional currency-Leg 2': 'USD',
  'Notional q